# Course Personalized Learning Insights Task

**Research:** An End-to-End Study on Prompt Engineering Techniques in Generative AI  
for Automated Course Understanding, Skill Extraction, and Personalized Learning Insights

**Setup and Execution Guide**

Step 1: Environment Setup

Install all required libraries and dependencies.

Step 2: Pipeline Configuration and Execution

This step covers dataset loading, prompt setup, model configuration, evaluation and exporting results.

CONFIG section at the top to be updated as per the need of experiments.

**Evaluation Metrics (7 core + latency + composite)**
- **Field Completeness** — fraction of the 6 required insight fields populated above minimum word count
- **BERTScore F1** — semantic similarity of full insight text against gold record (roberta-base)
- **ROUGE-L** — longest common subsequence recall between prediction and gold
- **Field-level BLEU** — corpus-level BLEU-2 across all six fields independently
- **Jaccard Similarity** — token-level set overlap between prediction and gold insight
- **Coherence Score** — average sentence count alignment across the six fields
- **Verbatim Copy Penalty** — fraction of fields that copy ≥10-word spans from the input description (lower = better; inverted for composite)
- **Latency** — API call latency in seconds (min / max / avg / total)
- **Composite Score** — weighted blend of all 7 metrics

In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================

import subprocess
import sys

SEPARATOR_WIDTH: int = 55

LIBRARY_INSTALLS = [
    ("openai>=1.30.0",),
    ("pandas>=2.0.0", "numpy>=1.24.0", "scipy>=1.10.0"),
    ("matplotlib>=3.7.0",),
    ("rouge-score>=0.1.2",),
    ("nltk>=3.8.0",),
]

BERT_INSTALLS = [
    ("torch",),
    ("transformers==4.35.2",),
    ("bert-score==0.3.13",),
]

VERSION_CHECK_LIBS: list = [
    ("openai",        "openai"),
    ("pandas",        "pandas"),
    ("numpy",         "numpy"),
    ("torch",         "torch"),
    ("transformers",  "transformers"),
    ("bert_score",    "bert_score"),
    ("matplotlib",    "matplotlib"),
    ("scipy",         "scipy"),
    ("rouge_score",   "rouge-score"),
    ("nltk",          "nltk"),
]


def pip_install(*packages: str) -> None:
    """Install one or more pip packages quietly via subprocess."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )


def print_library_versions(lib_map: list) -> None:
    """Print installed version for each library in lib_map.

    Args:
        lib_map: List of (import_name, display_name) tuples.
    """
    print("=" * SEPARATOR_WIDTH)
    print("INSTALLED LIBRARY VERSIONS")
    print("=" * SEPARATOR_WIDTH)
    for import_name, display_name in lib_map:
        try:
            mod = __import__(import_name)
            version = getattr(mod, "__version__", "unknown")
            print(f"  {display_name:<18} {version}")
        except ImportError:
            print(f"  {display_name:<18} NOT INSTALLED")
    print("=" * SEPARATOR_WIDTH)


print("[1/5] Installing core packages...")
for pkg_group in LIBRARY_INSTALLS:
    pip_install(*pkg_group)

print("[2/5] Installing PyTorch...")
pip_install(*BERT_INSTALLS[0])

print("[3/5] Installing Transformers (pinned 4.35.2)...")
pip_install(*BERT_INSTALLS[1])

print("[4/5] Installing BERTScore (pinned 0.3.13)...")
pip_install(*BERT_INSTALLS[2])

print("[5/5] Downloading NLTK tokenizer data...")
import nltk
nltk.download("punkt", quiet=True)

print()
print_library_versions(VERSION_CHECK_LIBS)
print()
print("All packages installed successfully.")
print("If Colab prompts a runtime restart, click Restart.")
print("After restarting, run Cell 2 directly — do NOT re-run Cell 1.")

[1/5] Installing core packages...
[2/5] Installing PyTorch...
[3/5] Installing Transformers (pinned 4.35.2)...
[4/5] Installing BERTScore (pinned 0.3.13)...
[5/5] Downloading NLTK tokenizer data...

INSTALLED LIBRARY VERSIONS
  openai             2.32.0
  pandas             2.2.2
  numpy              2.0.2
  torch              2.10.0+cpu
  transformers       4.35.2
  bert_score         0.3.12
  matplotlib         3.10.0
  scipy              1.16.3
  rouge-score        unknown
  nltk               3.9.1

All packages installed successfully.
If Colab prompts a runtime restart, click Restart.
After restarting, run Cell 2 directly — do NOT re-run Cell 1.


In [ ]:
# =============================================================================
# CELL 2 — END-TO-END PERSONALIZED LEARNING INSIGHTS EXPERIMENT
#
# Sections
# --------
#   A.  Imports & library version display
#   B.  *** TEMPLATE VARIABLES — edit here to create a new experiment ***
#   C.  Configuration
#   D.  Dataset load
#   E.  API key setup
#   F.  _build_prompt() — *** edit here for each prompt technique ***
#   G.  LLM caller
#   H.  Run experiment
#   I.  Insight parsing & evaluation metrics
#   J.  Full metrics table
#   K.  Final summary
#   L.  Save & download CSVs
# =============================================================================


# =============================================================================
# A.  IMPORTS & LIBRARY VERSION DISPLAY
# =============================================================================

import getpass
import math
import os
import random
import re
import time
import warnings
from datetime import datetime, timezone
from typing import Dict, List, Optional, Set, Tuple

import matplotlib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from google.colab import drive
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from openai import OpenAI
from rouge_score import rouge_scorer

try:
    from google.colab import files
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

BERT_AVAILABLE: bool = False
BERT_WARNED: bool = False
try:
    from bert_score import score as bert_score_fn
    BERT_AVAILABLE = True
except ImportError:
    pass

matplotlib.rcParams["figure.dpi"] = 110

SEPARATOR_SM:  int = 40
SEPARATOR_MD:  int = 55
SEPARATOR_LG:  int = 65
SEPARATOR_XL:  int = 80
SEPARATOR_XXL: int = 90
TABLE_ROW_WIDTH: int = 115

LIBRARY_VERSION_MAP: List[Tuple[str, str]] = [
    ("openai",        "openai"),
    ("pandas",        "pandas"),
    ("numpy",         "numpy"),
    ("torch",         "torch"),
    ("transformers",  "transformers"),
    ("bert_score",    "bert_score"),
    ("matplotlib",    "matplotlib"),
    ("scipy",         "scipy"),
    ("rouge_score",   "rouge-score"),
    ("nltk",          "nltk"),
]


def print_library_versions(lib_map: List[Tuple[str, str]]) -> None:
    """Print the active runtime version for each library.

    Args:
        lib_map: List of (import_name, display_name) tuples.
    """
    print("=" * SEPARATOR_MD)
    print("LIBRARY VERSIONS (active runtime)")
    print("=" * SEPARATOR_MD)
    for import_name, display_name in lib_map:
        try:
            mod = __import__(import_name)
            version = getattr(mod, "__version__", "unknown")
            print(f"  {display_name:<18} {version}")
        except ImportError:
            print(f"  {display_name:<18} NOT INSTALLED")
    print("=" * SEPARATOR_MD)
    print(f"  BERTScore available : {BERT_AVAILABLE}")
    print("=" * SEPARATOR_MD)


print_library_versions(LIBRARY_VERSION_MAP)


# =============================================================================
# B.  *** TEMPLATE VARIABLES ***
#
#  Edit only this block to create a new experiment variant.
#  --------------
#  TECHNIQUE        : "zero_shot" | "few_shot" | "instruction" | "role_based"
#                     "chain_of_thought" | "self_consistency" | "react" | "tree_of_thought"
#  GROQ_SECRET_NAME : the Colab Secret name that holds the API key for this slot
# =============================================================================

TECHNIQUE: str = "zero_shot"
TASK_TYPE: str = "personalized_learning_insights"
GROQ_SECRET_NAME: str = "GROQ_ZEROSHOT"


# =============================================================================
# C.  CONFIGURATION
# =============================================================================

# Number of repeated runs  (1 = quick test | 3-5 = full research study)
NUM_RUNS: int = 1

# Temperature (low for consistent structured output)
TEMPERATURE: float = 0.2

TOP_P: float = 0.9

MODELS: List[str] = [
    "llama-3.1-8b-instant",
    "qwen/qwen3-32b",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
]

GROQ_BASE_URL: str = "https://api.groq.com/openai/v1"

# 1024 tokens truncates the 6-field response; 2048 is the safe minimum.
MAX_TOKENS: int = 2048

DELAY_MIN: int = 7
DELAY_MAX: int = 12
N_ROWS: int = 25

# The six structured fields the model must populate.
# Field labels must match exactly what the prompt instructs the model to output.
INSIGHT_FIELDS: List[str] = [
    "target_learner",
    "prerequisite_skills",
    "skills_gained",
    "learning_outcomes",
    "career_relevance",
    "next_steps",
]

# Minimum words per field to count as non-trivial
MIN_FIELD_WORD_COUNT: int = 5


def print_experiment_config() -> None:
    """Print a summary of the current experiment configuration."""
    total_calls = NUM_RUNS * len(MODELS) * N_ROWS
    avg_delay_min = total_calls * (DELAY_MIN + DELAY_MAX) / 2 / 60
    print("Experiment Configuration")
    print("-" * SEPARATOR_SM)
    print(f"  Technique          : {TECHNIQUE}")
    print(f"  Task type          : {TASK_TYPE}")
    print(f"  Temperature        : {TEMPERATURE}")
    print(f"  Runs               : {NUM_RUNS}")
    print(f"  Models             : {MODELS}")
    print(f"  Courses            : {N_ROWS}")
    print(f"  Max tokens         : {MAX_TOKENS}")
    print(f"  Insight fields     : {len(INSIGHT_FIELDS)}")
    print(f"  Total API calls    : {total_calls}")
    print(f"  Est. delay time    : ~{avg_delay_min:.1f} min")
    print(f"  top_p              : {TOP_P}")


print_experiment_config()


# =============================================================================
# D.  DATASET LOAD
# =============================================================================

COLUMN_RENAME_MAP: Dict[str, str] = {
    "title":        "course_title",
    "Organization": "course_organization",
    "Level":        "course_difficulty",
    "Description":  "description",
    "enrolled":     "course_students_enrolled",
    "rating":       "course_rating",
}

TEXT_COLUMNS: List[str] = [
    "course_title",
    "course_organization",
    "course_difficulty",
    "description",
]

# Gold insight CSV columns expected after load
GOLD_INSIGHT_FIELDS: List[str] = [
    "gold_target_learner",
    "gold_prerequisite_skills",
    "gold_skills_gained",
    "gold_learning_outcomes",
    "gold_career_relevance",
    "gold_next_steps",
]


def load_dataset(n_rows: int = N_ROWS) -> pd.DataFrame:
    """Load, rename, and clean the Coursera CSV dataset from Google Drive.
    Also loads and merges the gold learning insights CSV for evaluation.

    The gold CSV is expected at:
        /content/drive/MyDrive/research/data/gold_learning_insights_coursera.csv

    Required gold columns (must match GOLD_INSIGHT_FIELDS):
        gold_target_learner, gold_prerequisite_skills, gold_skills_gained,
        gold_learning_outcomes, gold_career_relevance, gold_next_steps

    Join key: course_title (left join — unmatched courses get NaN gold fields).

    Args:
        n_rows: Maximum number of rows to load.

    Returns:
        Cleaned DataFrame with standardised columns and merged gold insight fields.

    Raises:
        FileNotFoundError: If either CSV is not found at the expected path.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("Dataset Load")
    print("-" * SEPARATOR_SM)
    print("  Loading Coursera dataset from Google Drive...")
    print("  Please wait...\n")

    drive.mount("/content/drive")

    dataset_path  = "/content/drive/MyDrive/research/data/coursera_clean_sample.csv"
    gold_csv_path = "/content/drive/MyDrive/research/data/gold_learning_insights_coursera.csv"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Dataset not found at '{dataset_path}'.\n"
            "Please ensure the file exists in your Google Drive."
        )

    print(f"  Found dataset at: {dataset_path}")

    df = pd.read_csv(dataset_path, encoding="latin-1")
    df = df.rename(columns=COLUMN_RENAME_MAP)
    df = df.dropna(subset=["course_title"]).head(n_rows).reset_index(drop=True)

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("N/A").astype(str)

    print(f"  Loaded {len(df)} courses successfully.")

    # Load gold insights
    if not os.path.exists(gold_csv_path):
        raise FileNotFoundError(
            f"Gold insights CSV not found at '{gold_csv_path}'.\n"
            "Generate it first using the Golden_Truth_Personalized_Learning_Insights notebook.\n"
            "Expected file: gold_learning_insights_coursera.csv in your Drive."
        )

    print(f"  Loading gold insights from: {gold_csv_path}")
    gold_df = pd.read_csv(gold_csv_path, encoding="latin-1")

    # Rename gold columns to prefixed form for unambiguous merging
    rename_gold: Dict[str, str] = {}
    for field in INSIGHT_FIELDS:
        if field in gold_df.columns:
            rename_gold[field] = f"gold_{field}"
    gold_df = gold_df.rename(columns=rename_gold)

    available_gold_cols = [
        c for c in GOLD_INSIGHT_FIELDS if c in gold_df.columns
    ]
    print(f"  Gold insight columns found : {available_gold_cols}")

    merge_cols = ["course_title"] + available_gold_cols
    if "course_title" in gold_df.columns:
        df = df.merge(gold_df[merge_cols], on="course_title", how="left")
    else:
        # Positional fallback if no course_title in gold file
        for col in available_gold_cols:
            df[col] = gold_df[col].values[: len(df)]

    for col in GOLD_INSIGHT_FIELDS:
        if col not in df.columns:
            df[col] = ""

    missing = df[GOLD_INSIGHT_FIELDS].isna().all(axis=1).sum()
    if missing:
        print(
            f"  Warning: {missing} course(s) have no gold insight fields — "
            "they will receive NaN scores in evaluation."
        )
    print(f"  Gold insights loaded for : {len(df) - missing} courses")
    return df


def get_course_context(row: pd.Series) -> Dict[str, str]:
    """Convert a dataset row into a context dict for prompt templates.

    Args:
        row: A single row from the dataset DataFrame.

    Returns:
        Dictionary with course fields plus gold insight fields.
    """
    ctx = {
        "title":        str(row.get("course_title", "Unknown")),
        "organization": str(row.get("course_organization", "N/A")),
        "difficulty":   str(row.get("course_difficulty", "Intermediate")),
        "description":  str(row.get("description", "No description available.")),
    }
    for gold_col in GOLD_INSIGHT_FIELDS:
        ctx[gold_col] = str(row.get(gold_col, "") or "")
    return ctx


DATASET_DF: pd.DataFrame = load_dataset(N_ROWS)
print("Loaded rows:", len(DATASET_DF))


# =============================================================================
# E.  API KEY SETUP
# =============================================================================

def load_groq_api_key(secret_name: str) -> str:
    """Load the Groq API key from Colab Secrets, falling back to getpass.

    Args:
        secret_name: The Colab Secret key name (e.g. "GROQ_ZEROSHOT").

    Returns:
        Groq API key string.

    Raises:
        ValueError: If no key is provided or found.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("API Key Setup")
    print("-" * SEPARATOR_SM)

    api_key = ""

    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get(secret_name).strip()
            print(f"  Loaded key from Colab Secret '{secret_name}'.")
        except Exception:
            print(
                f"  Secret '{secret_name}' not found in Colab Secrets.\n"
                "  Falling back to manual entry."
            )

    if not api_key:
        api_key = getpass.getpass(
            f"  Enter your Groq API key for '{secret_name}': "
        ).strip()

    if not api_key:
        raise ValueError(
            "No Groq API key provided. "
            f"Add it as a Colab Secret named '{secret_name}' "
            "or enter it when prompted."
        )

    print(f"  Key loaded : gsk_...{api_key[-4:]}")
    return api_key


GROQ_API_KEY: str = load_groq_api_key(GROQ_SECRET_NAME)


# =============================================================================
# F.  *** PROMPT ***
#
#  The ctx dict provides:
#    ctx["title"]        — course title
#    ctx["organization"] — course provider
#    ctx["difficulty"]   — Beginner / Intermediate / Advanced
#    ctx["description"]  — cleaned course description
#
#  Output format requirement:
#    The prompt MUST instruct the model to return exactly 6 labelled fields
#    using PLAIN UPPERCASE labels (no markdown bold, no hashes), e.g.:
#
#      TARGET_LEARNER: <value>
#      PREREQUISITE_SKILLS: <value>
#      SKILLS_GAINED: <value>
#      LEARNING_OUTCOMES: <value>
#      CAREER_RELEVANCE: <value>
#      NEXT_STEPS: <value>
#
#    This format is required for the parse_insight_response() function below.
# =============================================================================


def _build_prompt(ctx: Dict[str, str]) -> str:
    """Return the filled prompt string for this personalized learning insights experiment.

    Replace the body of this function with your own prompt technique.

    IMPORTANT:
    - The model must return exactly the 6 labelled fields shown below.
    - Do not use markdown bold (**LABEL**) or heading hashes (### LABEL).
    - Plain uppercase labels followed by a colon are required for parsing.

    Args:
        ctx: Course context dict from get_course_context().

    Returns:
        Fully rendered prompt string ready for the LLM.
    """
    return f"""Generate a structured personalized learning insight record for the following course.

            Use concise and course-specific content for every field.

            Output only the completed fields in the exact format shown below.

            Difficulty   : {ctx['difficulty']}
            Description  : {ctx['description']}

            Respond using EXACTLY these plain labels with no markdown formatting:

            TARGET_LEARNER:
            PREREQUISITE_SKILLS:
            SKILLS_GAINED:
            LEARNING_OUTCOMES:
            CAREER_RELEVANCE:
            NEXT_STEPS:
            """


print(f"\nPrompt template ready: {TECHNIQUE} / {TASK_TYPE}")


# =============================================================================
# G.  LLM CALLER
# =============================================================================

TPD_ERROR_PHRASES: Tuple[str, ...] = (
    "tokens per day",
    "tpd",
    "on_demand",
    "daily token",
    "token quota",
    "rate limit",
)

RETRY_BACKOFF_MIN: int = 15
RETRY_BACKOFF_MAX: int = 30


class TokenLimitExceeded(Exception):
    """Raised when the Groq daily token quota (TPD) is exhausted.

    No retry is attempted upon catching this exception.
    The caller is responsible for saving any partial results.
    """


def is_token_limit_error(exc: Exception) -> bool:
    """Return True if the exception signals a Groq daily TPD quota error.

    Args:
        exc: Exception raised by the OpenAI client.

    Returns:
        True if any TPD-related phrase is found in the error message.
    """
    return any(phrase in str(exc).lower() for phrase in TPD_ERROR_PHRASES)


def call_llm(
    prompt: str,
    model_name: str,
    max_tokens: int = MAX_TOKENS,
    top_p: float = 0.9,
    retries: int = 3,
) -> Tuple[str, float]:
    """Call the Groq API and return (response_text, latency_seconds).

    Uses the module-level GROQ_API_KEY and TEMPERATURE.
    Strips <think>...</think> blocks from Qwen3 chain-of-thought outputs.
    Raises TokenLimitExceeded immediately on daily quota errors.
    Retries up to `retries` times with random back-off on transient errors.

    Args:
        prompt:     Fully rendered user prompt string.
        model_name: Groq model identifier string.
        max_tokens: Maximum tokens in the completion.
        top_p:      Nucleus sampling parameter.
        retries:    Number of retry attempts on transient errors.

    Returns:
        Tuple of (cleaned_response_text, latency_in_seconds).

    Raises:
        TokenLimitExceeded: If a daily TPD quota error is detected.
    """
    client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)
    messages = [{"role": "user", "content": prompt}]

    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"      Waiting {delay:.0f}s...", end=" ", flush=True)
    time.sleep(delay)

    for attempt in range(1, retries + 1):
        t_start = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=TEMPERATURE,
                top_p=top_p,
                max_tokens=max_tokens,
            )
            latency = round(time.perf_counter() - t_start, 3)
            raw_text = response.choices[0].message.content or ""
            # Strip Qwen3 chain-of-thought blocks before returning
            clean = re.sub(
                r"<think>.*?</think>", "", raw_text, flags=re.DOTALL
            ).strip()
            output_text = clean if clean else raw_text.strip()
            print(f"OK ({latency}s | temp={TEMPERATURE})")
            return output_text, latency

        except Exception as exc:
            latency = round(time.perf_counter() - t_start, 3)
            if is_token_limit_error(exc):
                raise TokenLimitExceeded(str(exc)) from exc
            print(f"\n      Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                print(f"      Retrying in {backoff:.0f}s...")
                time.sleep(backoff)
            else:
                print("      All retries exhausted. Recording empty output.")
                return "", latency


print("LLM caller ready.")


# =============================================================================
# H.  RUN EXPERIMENT
# =============================================================================

PROMPT_SNIPPET_LENGTH: int = 300


def build_result_record(
    run: int,
    model_name: str,
    idx: int,
    ctx: Dict[str, str],
    output: str,
    latency: float,
    prompt: str,
) -> Dict:
    """Assemble a single result record dict for storage in the results list.

    Args:
        run:        Run number (1-indexed).
        model_name: Groq model identifier.
        idx:        Course row index in the dataset.
        ctx:        Course context dictionary.
        output:     LLM response text (raw 6-field insight string).
        latency:    API call latency in seconds.
        prompt:     Full prompt string (truncated for storage).

    Returns:
        Dictionary with all result fields.
    """
    prompt_clean = re.sub(r"\s+", " ", prompt).strip()
    record = {
        "run":             run,
        "model":           model_name,
        "task_type":       TASK_TYPE,
        "technique":       TECHNIQUE,
        "temperature":     TEMPERATURE,
        "top_p":           TOP_P,
        "course_idx":      idx,
        "course_title":    ctx["title"],
        "difficulty":      ctx["difficulty"],
        "description_ref": ctx["description"],
        "output":          output,
        "latency_seconds": latency,
        "prompt_snippet":  prompt_clean[:PROMPT_SNIPPET_LENGTH] + "...",
        "timestamp":       datetime.now(timezone.utc).isoformat(),
    }
    # Store gold fields in the record for metric computation
    for gold_col in GOLD_INSIGHT_FIELDS:
        record[gold_col] = ctx.get(gold_col, "")
    return record


def print_token_limit_summary(
    all_records: List[Dict], exc: TokenLimitExceeded
) -> None:
    """Print a structured summary when the daily token limit is hit.

    Args:
        all_records: Records collected before the limit was reached.
        exc:         The TokenLimitExceeded exception that was raised.
    """
    last = all_records[-1] if all_records else {}
    print("\n" + "!" * SEPARATOR_LG)
    print("*** TOKEN LIMIT REACHED — experiment stopped here ***")
    print(f"  Model             : {last.get('model', 'N/A')}")
    print(f"  Task Type         : {last.get('task_type', 'N/A')}")
    print(f"  Technique         : {last.get('technique', 'N/A')}")
    print(f"  Run               : {last.get('run', 'N/A')}")
    print(f"  Course            : {last.get('course_title', 'N/A')}")
    print(f"  Records collected : {len(all_records)}")
    print(f"  Error             : {exc}")
    print("!" * SEPARATOR_LG)


all_records: List[Dict] = []
token_limit_hit: bool = False
experiment_ts: str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\n" + "=" * SEPARATOR_LG)
print("STARTING EXPERIMENT")
print("=" * SEPARATOR_LG)

try:
    for run in range(1, NUM_RUNS + 1):
        print(f"\n{'=' * SEPARATOR_LG}")
        print(f"RUN {run} / {NUM_RUNS}")
        print(f"{'=' * SEPARATOR_LG}")
        print(f"\n  [Task: {TASK_TYPE.upper()}]  temperature={TEMPERATURE}")

        for model_name in MODELS:
            print(f"\n    Model: {model_name}")
            print(f"\n      Technique: {TECHNIQUE}")

            for idx, row in DATASET_DF.iterrows():
                ctx = get_course_context(row)
                prompt = _build_prompt(ctx)
                print(
                    f"      [{idx:02d}] {ctx['title'][:50]}...",
                    end=" ",
                )

                output, latency = call_llm(
                    prompt,
                    model_name,
                    max_tokens=MAX_TOKENS,
                    top_p=TOP_P,
                )

                all_records.append(
                    build_result_record(
                        run, model_name, idx, ctx, output, latency, prompt
                    )
                )

except TokenLimitExceeded as exc:
    token_limit_hit = True
    print_token_limit_summary(all_records, exc)

RESULTS_DF: pd.DataFrame = pd.DataFrame(all_records)
experiment_status = "STOPPED EARLY (token limit)" if token_limit_hit else "COMPLETE"
print(f"\nExperiment {experiment_status}. Total records: {len(RESULTS_DF)}")


# =============================================================================
# I.  INSIGHT PARSING & EVALUATION METRICS
# =============================================================================

# ---------------------------------------------------------------------------
# Composite score weights
#
# 7 core metrics + latency (latency excluded from composite; reported separately).
# Two weight sets: one when BERTScore is available, one when it is not.
# Weights sum to 1.0 in both cases.
#
# Metric              | With BERT | Without BERT  | Rationale
# --------------------|-----------|---------------|---------------------------------
# field_completeness  |   0.25    |    0.30       | Structural correctness
# bert_f1             |   0.40    |    N/A        | Strongest semantic signal
# rouge_l             |   0.20    |    0.40       | Surface overlap; reliable proxy
# field_bleu          |   0.15    |    0.30       | N-gram fluency per field
# ---------------------------------------------------------------------------

WEIGHTS_WITH_BERT: Dict[str, float] = {
    "avg_field_completeness": 0.25,
    "avg_bert_f1":            0.40,
    "avg_rouge_l":            0.20,
    "avg_field_bleu":         0.15,
}

WEIGHTS_WITHOUT_BERT: Dict[str, float] = {
    "avg_field_completeness": 0.30,
    "avg_rouge_l":            0.40,
    "avg_field_bleu":         0.30,
}

BERTSCORE_BATCH_SIZE: int = 8
BERTSCORE_CANDIDATE_MODELS: List[str] = [
    "roberta-base",
    "xlm-roberta-base",
    "bert-base-uncased",
]


# ---------------------------------------------------------------------------
# Parser — same markdown-resilient normaliser used in the gold truth notebook
# ---------------------------------------------------------------------------

FIELD_LABEL_MAP: Dict[str, str] = {
    "TARGET_LEARNER":      "target_learner",
    "PREREQUISITE_SKILLS": "prerequisite_skills",
    "SKILLS_GAINED":       "skills_gained",
    "LEARNING_OUTCOMES":   "learning_outcomes",
    "CAREER_RELEVANCE":    "career_relevance",
    "NEXT_STEPS":          "next_steps",
}


def _normalise_response(raw_text: str) -> str:
    """Strip markdown decorators Gemini/LLMs add around field labels.

    Handles: **LABEL**: / **LABEL:** / ### LABEL: / * LABEL:
    Leaves plain LABEL: unchanged.

    Args:
        raw_text: Raw string returned by the LLM.

    Returns:
        Normalised string with markdown stripped from label lines.
    """
    text = re.sub(r'^#{1,6}\s*', '', raw_text, flags=re.MULTILINE)
    text = re.sub(r'\*{1,2}([A-Z_]+)\*{1,2}\s*:', r'\1:', text)
    text = re.sub(r'^\*+\s*([A-Z_]+)\s*:', r'\1:', text, flags=re.MULTILINE)
    return text


def parse_insight_response(raw_text: str) -> Dict[str, str]:
    """Extract the six structured insight fields from a raw LLM response.

    Normalises markdown decorators first, then splits on bare LABEL: tokens.
    Missing fields are returned as empty strings.

    Args:
        raw_text: Raw string returned by the LLM.

    Returns:
        Dictionary mapping field names to their extracted values.
    """
    parsed: Dict[str, str] = {col: "" for col in FIELD_LABEL_MAP.values()}
    if not raw_text:
        return parsed

    normalised = _normalise_response(raw_text)
    label_pattern = "|".join(re.escape(lbl) for lbl in FIELD_LABEL_MAP)
    splitter = re.compile(rf"^({label_pattern})\s*:\s*", re.MULTILINE)

    parts = splitter.split(normalised)
    it = iter(parts[1:])   # skip preamble
    for label, value in zip(it, it):
        col_name = FIELD_LABEL_MAP.get(label.strip())
        if col_name:
            parsed[col_name] = value.strip()
    return parsed


# ---------------------------------------------------------------------------
# Metric 1 — Field Completeness
# ---------------------------------------------------------------------------

def field_completeness(parsed: Dict[str, str]) -> float:
    """Fraction of the 6 required fields that meet the minimum word count.

    A field is considered complete when its word count >= MIN_FIELD_WORD_COUNT.
    Score 1.0 means all 6 fields are populated; 0.0 means all are empty.

    Args:
        parsed: Dictionary of extracted field values.

    Returns:
        Completeness score in [0, 1] rounded to 4 decimal places.
    """
    complete = sum(
        1 for f in INSIGHT_FIELDS
        if len(parsed.get(f, "").split()) >= MIN_FIELD_WORD_COUNT
    )
    return round(complete / len(INSIGHT_FIELDS), 4)


# ---------------------------------------------------------------------------
# Metric 2 — BERTScore F1
# ---------------------------------------------------------------------------

def compute_bertscore(
    hypotheses: List[str], references: List[str]
) -> List[float]:
    """Compute BERTScore F1 for each (hypothesis, reference) pair.

    Each hypothesis and reference are the full concatenated insight texts.
    Tries candidate models in order; returns NaN per item if all fail.

    Args:
        hypotheses: List of predicted insight strings (fields joined).
        references: List of gold insight strings (fields joined).

    Returns:
        List of BERTScore F1 values, one per pair.
    """
    global BERT_WARNED
    if not BERT_AVAILABLE:
        if not BERT_WARNED:
            print("[BERTScore] Not available — bert_f1 will be NaN.")
            print("  Run Cell 1, restart runtime, then re-run Cell 2.")
            BERT_WARNED = True
        return [float("nan")] * len(hypotheses)

    for model_type in BERTSCORE_CANDIDATE_MODELS:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                _, _, f_scores = bert_score_fn(
                    hypotheses,
                    references,
                    lang="en",
                    model_type=model_type,
                    verbose=False,
                    batch_size=BERTSCORE_BATCH_SIZE,
                )
            print(f"[BERTScore] Model used: {model_type}")
            return [round(float(s), 4) for s in f_scores]
        except Exception as exc:
            print(f"[BERTScore] Failed with {model_type}: {exc}")

    print("[BERTScore] All models failed — bert_f1 = NaN.")
    return [float("nan")] * len(hypotheses)


# ---------------------------------------------------------------------------
# Metric 3 — ROUGE-L
# ---------------------------------------------------------------------------

_rouge_scorer_instance = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def compute_rouge_l(prediction: str, reference: str) -> float:
    """Compute ROUGE-L F1 between the full prediction and reference insight texts.

    Args:
        prediction: Full predicted insight text (all fields concatenated).
        reference:  Full gold insight text (all gold fields concatenated).

    Returns:
        ROUGE-L F1 score in [0, 1] rounded to 4 decimal places.
    """
    if not prediction.strip() or not reference.strip():
        return 0.0
    scores = _rouge_scorer_instance.score(reference, prediction)
    return round(scores["rougeL"].fmeasure, 4)


# ---------------------------------------------------------------------------
# Metric 4 — Field-level BLEU-2
# ---------------------------------------------------------------------------

def compute_field_bleu(
    parsed_pred: Dict[str, str],
    parsed_gold: Dict[str, str],
) -> float:
    """Compute corpus-level BLEU-2 averaged across all six insight fields.

    Each field is treated as an independent (hypothesis, reference) pair.
    Uses add-1 smoothing to handle short or empty fields gracefully.

    Args:
        parsed_pred: Parsed prediction fields (dict of field -> text).
        parsed_gold: Parsed gold fields (dict of gold_field -> text).

    Returns:
        Mean BLEU-2 score in [0, 1] rounded to 4 decimal places.
    """
    smoothie = SmoothingFunction().method1
    field_scores: List[float] = []

    for field in INSIGHT_FIELDS:
        pred_text = parsed_pred.get(field, "") or ""
        gold_text = parsed_gold.get(f"gold_{field}", "") or ""

        hyp_tokens = pred_text.lower().split()
        ref_tokens = gold_text.lower().split()

        if not hyp_tokens or not ref_tokens:
            field_scores.append(0.0)
            continue

        try:
            score = corpus_bleu(
                [[ref_tokens]],
                [hyp_tokens],
                weights=(0.5, 0.5),
                smoothing_function=smoothie,
            )
            field_scores.append(round(float(score), 4))
        except Exception:
            field_scores.append(0.0)

    return round(sum(field_scores) / len(field_scores), 4) if field_scores else 0.0


# ---------------------------------------------------------------------------
# Row-level metric computation
# ---------------------------------------------------------------------------

def compute_row_metrics(
    row: pd.Series,
    bert_f1_scores: List[float],
    row_index: int,
) -> Dict:
    """Compute all 7 evaluation metrics for a single result row.

    Args:
        row:            A row from RESULTS_DF.
        bert_f1_scores: Pre-computed BERTScore F1 values (one per row).
        row_index:      Index of this row in the BERTScore list.

    Returns:
        Dictionary of metric values for this row.
    """
    output      = str(row.get("output", "") or "")
    parsed_pred = parse_insight_response(output)

    # Build gold dict from the prefixed gold columns stored in the record
    parsed_gold: Dict[str, str] = {
        col: str(row.get(col, "") or "") for col in GOLD_INSIGHT_FIELDS
    }

    # Concatenate all fields for string-level metrics
    pred_full = " ".join(
        parsed_pred.get(f, "") for f in INSIGHT_FIELDS
    ).strip()
    gold_full = " ".join(
        parsed_gold.get(f"gold_{f}", "") for f in INSIGHT_FIELDS
    ).strip()

    return {
        "run":                row["run"],
        "model":              row["model"],
        "temperature":        TEMPERATURE,
        "course_idx":         row["course_idx"],
        # 7 core metrics
        "field_completeness": field_completeness(parsed_pred),
        "bert_f1":            bert_f1_scores[row_index],
        "rouge_l":            compute_rouge_l(pred_full, gold_full),
        "field_bleu":         compute_field_bleu(parsed_pred, parsed_gold),
        # Latency
        "latency_seconds":    row.get("latency_seconds", float("nan")),
    }


# ---------------------------------------------------------------------------
# Composite score
# ---------------------------------------------------------------------------

def compute_composite_score(
    agg: pd.DataFrame, bert_available: bool
) -> pd.Series:
    """Compute a weighted composite score from aggregated metric columns.

    copy_penalty is inverted (1 - penalty) before weighting so that
    lower copying yields a higher composite contribution.

    Args:
        agg:            Aggregated metrics DataFrame.
        bert_available: Whether BERTScore values are available.

    Returns:
        Series of composite scores rounded to 4 decimal places.
    """
    if bert_available:
        w = WEIGHTS_WITH_BERT
        return (
            agg["avg_field_completeness"]         * w["avg_field_completeness"]
            + agg["avg_bert_f1"].fillna(0)        * w["avg_bert_f1"]
            + agg["avg_rouge_l"]                  * w["avg_rouge_l"]
            + agg["avg_field_bleu"]               * w["avg_field_bleu"]
        ).round(4)
    else:
        w = WEIGHTS_WITHOUT_BERT
        return (
            agg["avg_field_completeness"]         * w["avg_field_completeness"]
            + agg["avg_rouge_l"]                  * w["avg_rouge_l"]
            + agg["avg_field_bleu"]               * w["avg_field_bleu"]
        ).round(4)


# ---------------------------------------------------------------------------
# Evaluate
# ---------------------------------------------------------------------------

def evaluate(results_df: pd.DataFrame) -> pd.DataFrame:
    """Compute and aggregate all evaluation metrics from the results DataFrame.

    Groups by (run, model) and computes mean values for each metric
    plus a weighted composite score.

    Args:
        results_df: Raw results DataFrame from the experiment loop.

    Returns:
        Aggregated metrics DataFrame sorted by composite_score descending.
    """
    if results_df.empty:
        print("No results to evaluate.")
        return pd.DataFrame()

    print("\nParsing predicted insights and computing BERTScore...")
    print("(downloads roberta-base ~500 MB on first run)")

    # Build concatenated strings for BERTScore
    hyp_strings: List[str] = []
    ref_strings: List[str] = []
    for _, row in results_df.iterrows():
        parsed = parse_insight_response(str(row.get("output", "") or ""))
        hyp_strings.append(
            " ".join(parsed.get(f, "") for f in INSIGHT_FIELDS)
        )
        ref_strings.append(
            " ".join(
                str(row.get(f"gold_{f}", "") or "") for f in INSIGHT_FIELDS
            )
        )

    bert_scores = compute_bertscore(hyp_strings, ref_strings)

    print("Computing row-level metrics...")
    row_metric_records: List[Dict] = [
        compute_row_metrics(row, bert_scores, i)
        for i, (_, row) in enumerate(results_df.iterrows())
    ]
    per_row_df = pd.DataFrame(row_metric_records)

    print("Aggregating...")
    agg = per_row_df.groupby(["run", "model"]).agg(
        avg_field_completeness=("field_completeness", "mean"),
        avg_bert_f1=           ("bert_f1",            "mean"),
        avg_rouge_l=           ("rouge_l",            "mean"),
        avg_field_bleu=        ("field_bleu",         "mean"),
        avg_latency_sec=       ("latency_seconds",    "mean"),
        min_latency_sec=       ("latency_seconds",    "min"),
        max_latency_sec=       ("latency_seconds",    "max"),
        total_latency_sec=     ("latency_seconds",    "sum"),
        total_samples=         ("course_idx",         "count"),
    ).round(4).reset_index()

    agg.insert(0, "technique",   TECHNIQUE)
    agg.insert(1, "task_type",   TASK_TYPE)
    agg.insert(2, "temperature", TEMPERATURE)

    bert_is_available = agg["avg_bert_f1"].notna().any()
    agg["composite_score"] = compute_composite_score(agg, bert_is_available)

    return agg.sort_values(
        "composite_score", ascending=False
    ).reset_index(drop=True)


METRICS_DF: pd.DataFrame = evaluate(RESULTS_DF)
print(f"Metrics computed: {len(METRICS_DF)} rows")


# =============================================================================
# J.  FULL METRICS TABLE
# =============================================================================

DISPLAY_COLUMNS: List[str] = [
    "run", "model", "task_type", "technique", "temperature",
    "avg_field_completeness", "avg_bert_f1",
    "avg_rouge_l", "avg_field_bleu", "avg_jaccard",
    "avg_latency_sec", "min_latency_sec", "max_latency_sec", "total_latency_sec",
    "composite_score",
]


def print_full_metrics_table(metrics_df: pd.DataFrame) -> None:
    """Configure pandas display and print the full metrics table.

    Args:
        metrics_df: Aggregated metrics DataFrame.
    """
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 260)
    pd.set_option("display.float_format", "{:.4f}".format)

    visible_cols = [
        col for col in DISPLAY_COLUMNS if col in metrics_df.columns
    ]
    print("\n" + "=" * SEPARATOR_XL)
    print("FULL METRICS TABLE — PERSONALIZED LEARNING INSIGHTS")
    print("=" * SEPARATOR_XL)
    print(metrics_df[visible_cols].to_string(index=False))


print_full_metrics_table(METRICS_DF)


# =============================================================================
# K.  FINAL SUMMARY — BEST & WORST MODEL
# =============================================================================

SUMMARY_METRIC_COLS: List[str] = [
    "avg_field_completeness", "avg_bert_f1",
    "avg_rouge_l", "avg_field_bleu", "avg_jaccard",
    "avg_latency_sec", "composite_score",
]


def build_best_worst_summary(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Build a DataFrame ranking best and worst models for this experiment.

    Averages composite_score across all runs before ranking.

    Args:
        metrics_df: Aggregated metrics DataFrame from evaluate().

    Returns:
        DataFrame with one BEST row and one WORST row.
    """
    if metrics_df.empty:
        return pd.DataFrame()

    valid_cols = [
        col for col in SUMMARY_METRIC_COLS if col in metrics_df.columns
    ]
    grouped = (
        metrics_df
        .groupby("model")[valid_cols]
        .mean()
        .round(4)
        .reset_index()
        .sort_values("composite_score", ascending=False)
        .reset_index(drop=True)
    )

    best  = grouped.iloc[0].to_dict()
    worst = grouped.iloc[-1].to_dict()
    best["rank"]  = "BEST"
    worst["rank"] = "WORST"

    summary = pd.DataFrame([best, worst])
    summary.insert(0, "technique",   TECHNIQUE)
    summary.insert(1, "task_type",   TASK_TYPE)
    summary.insert(2, "temperature", TEMPERATURE)
    return summary


def format_bert_value(row: pd.Series) -> str:
    """Format avg_bert_f1 for display, returning N/A if unavailable.

    Args:
        row: A row from the summary DataFrame.

    Returns:
        Formatted string or '  N/A  ' if NaN.
    """
    if "avg_bert_f1" in row.index and not math.isnan(row["avg_bert_f1"]):
        return f"{row['avg_bert_f1']:.4f}"
    return "  N/A  "


def print_best_worst_summary(summary_df: pd.DataFrame) -> None:
    """Print a formatted best-and-worst model summary table to stdout.

    Args:
        summary_df: DataFrame produced by build_best_worst_summary().
    """
    if summary_df.empty:
        print("No data for best/worst summary.")
        return

    print("\n" + "=" * SEPARATOR_XXL)
    print("FINAL SUMMARY — BEST & WORST MODEL (PERSONALIZED LEARNING INSIGHTS)")
    print(f"Technique: {TECHNIQUE}  |  Task: {TASK_TYPE}  |  Temperature: {TEMPERATURE}")
    print("(composite_score averaged across all runs; higher = better)")
    print("=" * SEPARATOR_XXL)
    print(
        f"  {'Rank':<6} {'Model':<32} {'Composite':>9} {'Complete':>9} "
        f"{'BERT_F1':>8} {'ROUGE-L':>8} {'BLEU-2':>7} "
        f"{'Latency':>8}"
    )
    print("  " + "-" * TABLE_ROW_WIDTH)

    for _, row in summary_df.iterrows():
        bert_str    = format_bert_value(row)
        rank_marker = ">>>" if row["rank"] == "BEST" else "   "
        print(
            f"  {rank_marker} {row['rank']:<3} "
            f"{row['model']:<32} "
            f"{row['composite_score']:>9.4f} "
            f"{row['avg_field_completeness']:>9.4f} "
            f"{bert_str:>8} "
            f"{row['avg_rouge_l']:>8.4f} "
            f"{row['avg_field_bleu']:>7.4f} "
            f"{row['avg_latency_sec']:>8.3f}s"
        )

    print("\n" + "=" * SEPARATOR_XXL)
    print("KEY")
    print("  >>> BEST      = highest composite_score across all models")
    print("      WORST     = lowest composite_score across all models")
    print("  Composite     = weighted blend of all 4 metrics (higher is better)")
    print("  Complete      = fraction of 6 insight fields populated above minimum word count")
    print("  BERT_F1       = semantic similarity of full insight vs gold (roberta-base)")
    print("  ROUGE-L       = longest common subsequence F1 vs gold insight")
    print("  BLEU-2        = corpus BLEU-2 averaged across all 6 fields independently")
    print("  Latency       = avg API call latency in seconds")
    print("=" * SEPARATOR_XXL)


SUMMARY_DF: pd.DataFrame = build_best_worst_summary(METRICS_DF)
print_best_worst_summary(SUMMARY_DF)


# =============================================================================
# L.  SAVE & DOWNLOAD
# =============================================================================

_file_tag: str = f"{TECHNIQUE}_{TASK_TYPE}_{experiment_ts}"
RESULTS_FILENAME: str = f"insights_results_{_file_tag}.csv"
METRICS_FILENAME: str = f"insights_metrics_{_file_tag}.csv"
SUMMARY_FILENAME: str = f"insights_best_worst_{_file_tag}.csv"


def save_results(
    results_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    in_colab: bool,
) -> None:
    """Save all result DataFrames to CSV and optionally trigger downloads.

    Files produced:
      insights_results_<tag>.csv    — raw LLM outputs + gold fields
      insights_metrics_<tag>.csv    — aggregated metrics per (run, model)
      insights_best_worst_<tag>.csv — best and worst model summary

    Args:
        results_df: Raw results from the experiment loop.
        metrics_df: Aggregated metrics DataFrame.
        summary_df: Best/worst summary DataFrame.
        in_colab:   True if running inside Google Colab.
    """
    results_df.to_csv(RESULTS_FILENAME, index=False)
    metrics_df.to_csv(METRICS_FILENAME, index=False)
    if not summary_df.empty:
        summary_df.to_csv(SUMMARY_FILENAME, index=False)

    print(f"\nSaved: {RESULTS_FILENAME}  ({len(results_df)} rows)")
    print(f"Saved: {METRICS_FILENAME}  ({len(metrics_df)} rows)")
    print(f"Saved: {SUMMARY_FILENAME}  ({len(summary_df)} rows)")

    if in_colab:
        print("\nDownloading files...")
        files.download(RESULTS_FILENAME)
        files.download(METRICS_FILENAME)
        files.download(SUMMARY_FILENAME)
        print("Downloads initiated.")
    else:
        print("\nFiles saved locally.")

    print("\nDone.")


save_results(RESULTS_DF, METRICS_DF, SUMMARY_DF, IN_COLAB)

LIBRARY VERSIONS (active runtime)
  openai             2.32.0
  pandas             2.2.2
  numpy              2.0.2
  torch              2.10.0+cpu
  transformers       4.35.2
  bert_score         0.3.12
  matplotlib         3.10.0
  scipy              1.16.3
  rouge-score        unknown
  nltk               3.9.1
  BERTScore available : True
Experiment Configuration
----------------------------------------
  Technique          : zero_shot
  Task type          : personalized_learning_insights
  Temperature        : 0.2
  Runs               : 1
  Models             : ['llama-3.1-8b-instant', 'qwen/qwen3-32b', 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'llama-3.3-70b-versatile']
  Courses            : 25
  Max tokens         : 2048
  Insight fields     : 6
  Total API calls    : 125
  Est. delay time    : ~19.8 min
  top_p              : 0.9

----------------------------------------
Dataset Load
----------------------------------------
  Loading Coursera dataset from Google Drive...
  

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[BERTScore] Model used: roberta-base
Computing row-level metrics...
Aggregating...
Metrics computed: 5 rows

FULL METRICS TABLE — PERSONALIZED LEARNING INSIGHTS
 run                   model                      task_type technique  temperature  avg_field_completeness  avg_bert_f1  avg_rouge_l  avg_field_bleu  avg_latency_sec  min_latency_sec  max_latency_sec  total_latency_sec  composite_score
   1    llama-3.1-8b-instant personalized_learning_insights zero_shot       0.2000                  1.0000       0.8745       0.2950          0.1150           1.1045           0.3950           7.9250            27.6120           0.6760
   1     openai/gpt-oss-120b personalized_learning_insights zero_shot       0.2000                  1.0000       0.8702       0.2727          0.0842           1.5459           1.0230           5.4470            38.6480           0.6652
   1          qwen/qwen3-32b personalized_learning_insights zero_shot       0.2000                  1.0000       0.8697       0.269

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads initiated.

Done.
